In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))


# LSTM 往復バー予測モデル（lstm_clean）学習ノートブック

OCOブレイクアウト戦略 Phase A: クリーンブレイク vs 往復 の分類

ラベル（pearless-aux-labels dataset の y_clean_*）:
- 0 = 往復（次の5分足で P0±2.5銭 の両側に到達。OCOの構造的負けパターン）
- 1 = クリーン（片側のみ到達）
- 2 = 不到達（OCO不成立。学習から除外）

- 入力X: `/kaggle/input/pearless-usdjpy-m5/`（共通 npy）
- ラベル: `/kaggle/input/pearless-aux-labels/`
- 出力: `/kaggle/working/best_model.pt`

In [ ]:
import subprocess, sys

# Kaggle プリインストールの torch (2.10+cu128) は P100 (sm_60) のカーネルイメージを
# 含まず "CUDA error: no kernel image is available" で学習が落ちる（2026-06-11 実測）。
# バージョン無指定の `pip install torch` は既存の 2.10 で要件充足となり何もしないため、
# sm_60 / sm_75 両対応の torch==2.5.1+cu121 を明示的に固定インストールする。
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
import torch
assert torch.__version__.startswith("2.5.1"), f"torch downgrade failed: {torch.__version__}"
print(f"torch {torch.__version__}")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

# Kaggle 環境では /kaggle/input/{dataset-slug}/ にデータが配置される
DATASET_DIR = Path("/kaggle/input/datasets/nomuhosokawa/pearless-usdjpy-m5")
WORKING_DIR = Path("/kaggle/working")

# ローカル環境でのフォールバック（テスト実行時）
if not DATASET_DIR.exists():
    DATASET_DIR = Path("data")

# ソースコードを sys.path に追加
# ブラウザ追加時: /kaggle/input/pearless-src
# CLI dataset_sources時: /kaggle/input/datasets/nomuhosokawa/pearless-src
for _repo_candidate in [
    Path("/kaggle/input/pearless-src"),
    Path("/kaggle/input/datasets/nomuhosokawa/pearless-src"),
]:
    if _repo_candidate.exists():
        REPO_ROOT = _repo_candidate
        break
else:
    REPO_ROOT = Path("..")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Working dir: {WORKING_DIR}")

# 補助ラベル dataset（pearless-aux-labels）のパス解決
for _aux in [
    Path("/kaggle/input/pearless-aux-labels"),
    Path("/kaggle/input/datasets/nomuhosokawa/pearless-aux-labels"),
]:
    if _aux.exists():
        AUX_DIR = _aux
        break
else:
    AUX_DIR = Path("data/labels")
print(f"Aux labels dir: {AUX_DIR}")

In [ ]:
# データ読み込み（AC-013: Kaggle Dataset から numpy 配列をロード）
X_train: np.ndarray = np.load(DATASET_DIR / "X_train.npy")
y_train: np.ndarray = np.load(AUX_DIR / "y_clean_train.npy")
X_val: np.ndarray = np.load(DATASET_DIR / "X_val.npy")
y_val: np.ndarray = np.load(AUX_DIR / "y_clean_val.npy")

print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"y_train shape: {y_train.shape}, dtype: {y_train.dtype}")
print(f"X_val shape:   {X_val.shape},   dtype: {X_val.dtype}")
print(f"y_val shape:   {y_val.shape},   dtype: {y_val.dtype}")

# shape 検証（AC-002）
assert X_train.ndim == 3 and X_train.shape[1] == 60 and X_train.shape[2] == 16, (
    f"X_train shape 不正: {X_train.shape}"
)
assert X_val.ndim == 3 and X_val.shape[1] == 60 and X_val.shape[2] == 16, (
    f"X_val shape 不正: {X_val.shape}"
)

# クラス分布確認
unique, counts = np.unique(y_train, return_counts=True)
print("\nクラス分布 (y_train):")
for cls, cnt in zip(unique, counts):
    label = ["WHIPSAW", "CLEAN", "NOTOUCH"][cls]
    print(f"  {label}({cls}): {cnt} ({cnt / len(y_train) * 100:.1f}%)")

In [ ]:
# Phase A 用フィルタ: 往復(0) / クリーン(1) のみ残す（不到達(2) を除外）
train_mask = y_train != 2
val_mask = y_val != 2
X_train, y_train = X_train[train_mask], y_train[train_mask]
X_val, y_val = X_val[val_mask], y_val[val_mask]

import numpy as np
for name, y in [("y_train", y_train), ("y_val", y_val)]:
    unique, counts = np.unique(y, return_counts=True)
    dist = {int(u): int(c) for u, c in zip(unique, counts)}
    print(f"{name}: n={len(y)}, クラス分布={dist}")
assert set(np.unique(y_train)) == {0, 1}, "往復/クリーン 以外が残っている"


In [ ]:
# スモークテスト: 少量データ・少エポックで動作確認
# 本番学習時はこのセルをスキップ or SMOKE_TEST = False に変更
SMOKE_TEST = False

if SMOKE_TEST:
    X_train = X_train[:500]
    y_train = y_train[:500]
    X_val   = X_val[:100]
    y_val   = y_val[:100]
    print("[SMOKE] データを縮小: X_train", X_train.shape, "X_val", X_val.shape)


In [ ]:
# 往復バー予測 LSTM の初期化（MODEL_CONFIGS の設定に従う）
from models.configs import MODEL_CONFIGS

MODEL_NAME = "lstm_clean"
config = MODEL_CONFIGS[MODEL_NAME]
print(f"使用特徴量 ({config.n_features}): {list(config.features)}")
print(f"学習ハイパラ: {config.train}")

model = config.build_model()

# パラメータ数確認（≤ 10M）
total_params = sum(p.numel() for p in model.parameters())
assert total_params <= 10_000_000, f"パラメータ数が 10M 超: {total_params:,}"
print(f"パラメータ数: {total_params:,} (≤ 10M OK)")

# forward shape 動作確認
model.eval()
with torch.no_grad():
    x_test = torch.randn(4, 60, config.n_features)
    output = model(x_test)
assert output.shape == (4, 3), f"shape 不正: {output.shape}"
assert torch.allclose(output.sum(dim=1), torch.ones(4), atol=1e-4), (
    "softmax 合計が 1 でない"
)
assert output.min() >= 0.0, "確率が負値"
print(f"forward契約 OK: output.shape={output.shape}, softmax sum≈1.0")

In [ ]:
# 学習実行（AC-013: /kaggle/working/ にチェックポイント保存）
# 特徴量選択（config.features）は train_from_config 内で行うため、
# X_train / X_val はフル特徴量（16列）のまま渡す
from models.training import train_from_config

model = train_from_config(
    config,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    checkpoint_dir=str(WORKING_DIR),
    log_dir=str(WORKING_DIR / "logs"),
    n_epochs=3 if SMOKE_TEST else None,
    patience=3 if SMOKE_TEST else None,
)

In [ ]:
# チェックポイント保存確認（AC-013/014）
best_model_path = WORKING_DIR / "best_model.pt"
assert best_model_path.exists(), f"best_model.pt が存在しない: {best_model_path}"
print(f"AC-013 OK: チェックポイント保存確認: {best_model_path}")

# 保存したモデルをロードして動作確認（config と同一構成で再構築）
model_loaded = config.build_model()
model_loaded.load_state_dict(torch.load(best_model_path, map_location="cpu"))
model_loaded.eval()

with torch.no_grad():
    x_verify = torch.randn(2, 60, config.n_features)
    output_verify = model_loaded(x_verify)

assert output_verify.shape == (2, 3), f"ロード後 shape 不正: {output_verify.shape}"
print(f"AC-014 OK: モデルロード後の推論確認 shape={output_verify.shape}")

# ログファイル確認（AC-022）
log_dir = WORKING_DIR / "logs"
log_files = list(log_dir.glob(f"training_log_{MODEL_NAME}_*.csv"))
assert len(log_files) > 0, f"ログファイルが見つからない: {log_dir}"
print(f"AC-022 OK: ログファイル確認: {log_files[0]}")

print("\n全 AC 確認完了。Kaggle commit mode 実行成功。")